# Polynomial Regression으로 Calories_Burned 예측
주요 변경/추가 내용 확인해주세요
* Temp_diff 파생변수 추가 : 체온은 “절대값”보다 정상 체온(98.6)에서 얼마나 벗어났는지가 운동 강도와 더 직접 연결될 수 있음.
* 성별 분리 (M/F): 남/여 관계식이 달라서, 단일 모델보다 두 모델이 유리
* bestcols = (Exercise_Duration, BPM, Temp_diff, Age, Weight(lb))
* Polynomial degree + Ridge Polynomial로 곱(상호작용) 구조 학습-Calories는Weight×Duration×BPM 같은 상호작용이 핵심
* 정수 타겟에 맞춘 round RMSE 최적화-타겟이 정수이므로, round된 값이 실제 지표에 더 직접적으로 맞음
최종 RMSE : 0.13366

리더보드 : 0.12

In [2]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


/Users/admin/Desktop/DB/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 데이터 로드

In [3]:
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")
sub   = pd.read_csv("sample_submission.csv")

# 피처 엔지니어링

In [ ]:
# Temp_diff 파생변수 추가 : 체온은 “절대값”보다 정상 체온(98.6)에서 얼마나 벗어났는지가 운동 강도와 더 직접 연결될 수 있음.
def add_features(df):
    df = df.copy()  # 원본 데이터프레임을 수정하지 않도록 복사본 생성
    df["Temp_diff"] = df["Body_Temperature(F)"] - 98.6   # 체온 절댓값보다 정상체온(98.6°F)과의 편차가 운동 강도를 더 잘 반영함
    # 예: 체온 101.6 → Temp_diff = 3.0 (정상보다 3도 높음)
    return df

train = add_features(train) # train에 Temp_diff 파생변수 추가
test  = add_features(test)  # test에도 동일하게 적용 (train fit 불필요 — 상수 뺄셈이라 누수 없음

# OOF 기반 제거 실험을 반복한 결과 핵심피처 + Temp_diff 
# (Temp_diff = 체온 편차, 정상체온 98.6°F 기준으로 직접 계산한 파생변수)
COLS = ["Exercise_Duration", "BPM", "Temp_diff", "Age", "Weight(lb)"]
y    = train["Calories_Burned"].values.astype(float)

m_tr = (train["Gender"] == "M").values;    # train 남성 인덱스 (True/False 배열)
f_tr = ~m_tr     # train 여성 인덱스 (남성 마스크의 반전)
m_te = (test["Gender"]  == "M").values;  # test 남성 인덱스
f_te = ~m_te    # test 여성 인덱스



In [ ]:

def rmse(a, b): # RMSE 계산 함수 (위에서 정의한 것과 동일, 재정의)
    return np.sqrt(mean_squared_error(a, b))

def round_clip(pred): # 타겟 정수화용 예측값을 정수로 반올림하고 음수 방지를 위해 0 미만은 0으로 클리핑
    # 타겟(Calories_Burned)이 정수이므로 반올림이 실제 지표에 직접적으로 유리
    # round()는 파이썬 내장 함수 0.5일 때 가장 가까운 짝수로 반올림
    # np.rint()는 numpy 함수 numpy 배열 전체에 한번에 적용
    # 아래 셀 예측값 pred가 numpy 배열이라서 round(pred)는 배열 전체에 적용이 안 되고 np.rint(pred)는 벡터 연산으로 한번에 처리됨
    return np.rint(np.clip(pred, 0, None))

kf = KFold(n_splits=5, shuffle=True, random_state=42)
# 5-겹 교차검증 설정 (shuffle=True로 데이터 순서 영향 제거, 재현성을 위해 seed 고정)

In [ ]:
# KFold OOF 예측
# 각 fold에서 train fold로만 poly/scaler/모델을 fit
# validation fold는 transform만 수행해 데이터 누수를 방지
def oof_predict(X, y, degree, alpha):  
    oof = np.zeros(len(y))  # OOF 예측을 저장할 빈 배열 초기화
    for tr_idx, va_idx in kf.split(X):  # 5개 fold로 분할
        poly = PolynomialFeatures(degree=degree, include_bias=False)
        # include_bias=False: 상수항(1) 컬럼 제외 — Ridge가 자체적으로 intercept를 학습하므로 불필요
        sc   = StandardScaler()  # 피처 스케일 정규화 (평균 0, 표준편차 1)
        mdl  = Ridge(alpha=alpha, random_state=42)   # L2 규제 Ridge 회귀, alpha가 클수록 규제 강함

        # train fold: poly → scaler 순으로 fit_transform (fit과 transform 동시 수행)
        Xtr = sc.fit_transform(poly.fit_transform(X[tr_idx]))
        # validation fold: train fold에서 fit된 poly, scaler로 transform만 수행 (누수 방지)
        Xva = sc.transform(poly.transform(X[va_idx]))

        mdl.fit(Xtr, y[tr_idx])  # train fold로만 모델 학습
        oof[va_idx] = mdl.predict(Xva)  # validation fold 예측값을 OOF 배열에 저장
    return oof # 반환값: train 전체 길이의 누수 없는 OOF 예측 배열

# 성별별 최적 그리드 탐색
degrees = [2, 3, 4, 5]  # 다항식 차수 후보 (높을수록 표현력좋으나 과적합 위험도 높아짐)
alphas  = np.logspace(-6, 1, 18)    # 규제 강도 후보 18개 (1e-6 ~ 10, 로그 스케일)

best = {}   # 성별별 최적 (degree, alpha) 저장 딕셔너리
for gender, mask in [("M", m_tr), ("F", f_tr)]: # 남성/여성 각각 탐색
    Xg, yg = train.loc[mask, COLS].values.astype(float), y[mask]    # 해당 성별 데이터 추출

    best_score, best_deg, best_alpha = 1e18, None, None # 최솟값 추적 변수 초기화
    for deg in degrees:
        for alpha in alphas:
            # OOF 예측 후 반올림 → round RMSE 계산 (실제 제출 지표와 동일한 기준으로 탐색)
            score = rmse(yg, round_clip(oof_predict(Xg, yg, deg, alpha)))
            if score < best_score:  # 더 낮은 RMSE가 나오면 갱신
                best_score, best_deg, best_alpha = score, deg, alpha

    best[gender] = (best_deg, best_alpha)   # 해당 성별 최적값 저장
    print(f"[{gender}] degree={best_deg} | alpha={best_alpha:.4g} | OOF RMSE={best_score:.6f}")


# 최적 파라미터로 전체 OOF 예측 재생성 후 전체 RMSE 확인
oof_all = np.zeros(len(train))  # 전체 train에 대한 OOF 저장 배열
for gender, mask in [("M", m_tr), ("F", f_tr)]:
    deg, alpha = best[gender]   # 해당 성별 최적 파라미터 불러오기
    oof_all[mask] = oof_predict(train.loc[mask, COLS].values.astype(float), y[mask], deg, alpha)

print(f"\n전체 OOF RMSE: {rmse(y, round_clip(oof_all)):.6f}")   # 남성+여성 합산 최종 OOF RMSE


# 테스트 예측 (전체 train으로 재학습)
# OOF 탐색에서 찾은 최적 파라미터를 사용,train 전체를 학습에 활용해 모델 성능 최대화
def full_predict(X_tr, y_tr, X_te, degree, alpha):
    poly = PolynomialFeatures(degree=degree, include_bias=False) # 다항 피처 생성기
    sc   = StandardScaler() # 스케일러
    mdl  = Ridge(alpha=alpha, random_state=42)  # Ridge 모델
    # train 전체로 fit_transform → 모델 학습
    mdl.fit(sc.fit_transform(poly.fit_transform(X_tr)), y_tr)
    # test는 train에서 fit된 poly, scaler로 transform만 수행 (데이터 누수 방지)
    return mdl.predict(sc.transform(poly.transform(X_te)))

test_pred = np.zeros(len(test)) # test 예측값을 채울 빈 배열 초기화 (남성/여성 순서대로 채워짐)
for gender, tr_mask, te_mask in [("M", m_tr, m_te), ("F", f_tr, f_te)]:
    deg, alpha = best[gender]   # 해당 성별 최적 파라미터
    test_pred[te_mask] = full_predict(
        train.loc[tr_mask, COLS].values.astype(float), y[tr_mask],  # 해당 성별 train 데이터
        test.loc[te_mask, COLS].values.astype(float),   # 해당 성별 test 데이터
        deg, alpha
    )

[M] degree=2 | alpha=1e-06 | OOF RMSE=0.136973
[F] degree=2 | alpha=0.0007627 | OOF RMSE=0.130310

전체 OOF RMSE: 0.133666


# 제출용

In [ ]:

sub["Calories_Burned"] = round_clip(test_pred).astype(int) # 예측값 반올림 후 정수로 변환해 제출 컬럼에 저장
sub.to_csv("submit_gender_best.csv", index=False)
print("\n저장 완료: submit_gender_best.csv")


## 최종 RMSE : 0.1336


#  Final Model Summary

- Feature Set:
  ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight(lb)']

- 성별 분리 (M/F 각각 models)

- Polynomial Degree:
  M: 2
  F: 2

- Ridge Alpha:
  M: 1e-06
  F: 0.000762699

- Evaluation Metric:
  RMSE (after rounding)

- Final LB Score:
  0.12382

  데이터 구조에 맞게 성별을 분리하고, 물리적으로 해석 가능한 핵심 변수만 남긴 후, 2차 다항식 기반 Ridge 회귀로 함수 공간을 확장하고, 최종적으로 정수 예측 특성을 반영해 일반화 RMSE를 최적화한 모델입니다